# 09. LLM App Identity Patterns

## Background

LLM applications introduce new identity challenges that traditional OAuth and RBAC were not designed for. When an LLM agent calls a tool, it is acting on behalf of a user — but the agent itself is also a principal with its own identity and authorization scope. Who authorized this tool call? With what scope? On behalf of which user? These questions are unanswered in most LLM frameworks.

The emerging pattern is **OAuth-scoped tool calls**: the LLM agent holds an OAuth token scoped to exactly the permissions the user granted, and every tool call is checked against that token's scopes. This mirrors how mobile apps use OAuth — the app acts on behalf of the user, but only within the granted scope, and the user can revoke access at any time.

## What You'll Learn

- Tool authorization: checking OAuth scopes before executing LLM tool calls
- Per-user context isolation: each user session gets its own token and permission scope
- Scope minimization: agents request least-privilege scopes for their task
- Token binding: preventing token reuse across users or sessions
- Audit trail: logging every tool call with the authorizing token

## Why This Matters

Tool-calling LLM agents are rapidly moving from demos to production. The identity model for tool calls is still largely unspecified in the industry. This notebook implements the pattern that OWASP's LLM security guidance and the emerging MCP authorization spec recommend: OAuth tokens scoped to tool permissions, per-user isolation, and complete audit logging.

In [ ]:
import secrets, time, json, hashlib, hmac
from dataclasses import dataclass, field
from typing import Optional

print('LLM identity patterns demo ready')


## 1. OAuth-Scoped Tool Authorization

In [ ]:
TOOL_SCOPE_MAP = {
    'read_calendar':      'calendar:read',
    'write_calendar':     'calendar:write',
    'read_email':         'email:read',
    'send_email':         'email:send',
    'search_web':         'web:search',
    'read_crm':           'crm:read',
    'write_crm':          'crm:write',
    'execute_sql':        'db:execute',
    'read_files':         'files:read',
    'delete_files':       'files:delete',
}

@dataclass
class AgentToken:
    user_id:     str
    session_id:  str
    scopes:      set
    issued_at:   float
    expires_at:  float
    token_id:    str = field(default_factory=lambda: secrets.token_urlsafe(16))

    @property
    def valid(self): return time.time() < self.expires_at

    def has_scope(self, scope): return scope in self.scopes


class ToolAuthorizationLayer:
    def __init__(self):
        self._audit: list = []

    def authorize_tool_call(self, tool_name: str, token: AgentToken,
                             kwargs: dict = None) -> tuple:
        required_scope = TOOL_SCOPE_MAP.get(tool_name)
        entry = {
            'ts': time.time(), 'tool': tool_name,
            'user': token.user_id, 'session': token.session_id,
            'token_id': token.token_id, 'decision': None, 'reason': None,
        }
        if not token.valid:
            entry['decision'] = 'DENY'; entry['reason'] = 'Token expired'
            self._audit.append(entry)
            return False, 'Token expired'
        if required_scope is None:
            entry['decision'] = 'DENY'; entry['reason'] = f'Unknown tool: {tool_name}'
            self._audit.append(entry)
            return False, f'Unknown tool: {tool_name}'
        if not token.has_scope(required_scope):
            entry['decision'] = 'DENY'
            entry['reason']   = f'Missing scope: {required_scope}'
            self._audit.append(entry)
            return False, f'Tool {tool_name} requires scope {required_scope}'
        entry['decision'] = 'ALLOW'; entry['reason'] = f'Scope {required_scope} granted'
        self._audit.append(entry)
        return True, 'Authorized'

    def audit_log(self, n=10):
        return self._audit[-n:]


tool_auth = ToolAuthorizationLayer()

# User grants limited scopes for a calendar assistant
alice_token = AgentToken(
    user_id='alice', session_id=secrets.token_urlsafe(8),
    scopes={'calendar:read', 'calendar:write', 'email:read'},
    issued_at=time.time(), expires_at=time.time()+3600
)

tool_calls = [
    ('read_calendar',  {}),
    ('write_calendar', {'event': 'Standup'}),
    ('send_email',     {'to': 'bob@example.com'}),  # not in scope
    ('execute_sql',    {'query': 'SELECT *...'}),    # not in scope
    ('search_web',     {'query': 'weather'}),        # not in scope
]

print(f'Agent token scopes: {sorted(alice_token.scopes)}')
print()
for tool, kwargs in tool_calls:
    ok, reason = tool_auth.authorize_tool_call(tool, alice_token, kwargs)
    print(f'  [{"ALLOW" if ok else "DENY "}] {tool:<20} — {reason}')


## 2. Per-User Context Isolation

Each user session must be completely isolated. An agent handling multiple users must not allow token or context bleed between sessions.

In [ ]:
class MultiUserAgentSession:
    def __init__(self):
        self._sessions: dict = {}
        self._tool_auth = ToolAuthorizationLayer()

    def create_session(self, user_id: str, granted_scopes: set) -> str:
        session_id = secrets.token_urlsafe(16)
        token = AgentToken(
            user_id=user_id, session_id=session_id,
            scopes=granted_scopes,
            issued_at=time.time(), expires_at=time.time()+3600,
        )
        self._sessions[session_id] = {'token': token, 'history': []}
        return session_id

    def invoke_tool(self, session_id: str, tool: str, kwargs: dict = None):
        session = self._sessions.get(session_id)
        if session is None:
            return False, 'Invalid session'
        ok, reason = self._tool_auth.authorize_tool_call(tool, session['token'], kwargs)
        if ok:
            session['history'].append({'tool': tool, 'kwargs': kwargs, 'ts': time.time()})
        return ok, reason

    def session_summary(self, session_id: str):
        s = self._sessions.get(session_id)
        if not s: return None
        return {'user': s['token'].user_id, 'scopes': sorted(s['token'].scopes),
                'tool_calls': len(s['history'])}


agent = MultiUserAgentSession()
sess_alice = agent.create_session('alice', {'calendar:read','calendar:write','email:read'})
sess_bob   = agent.create_session('bob',   {'crm:read','web:search'})

print('Alice session:')
for tool in ['read_calendar','read_email','read_crm']:
    ok, reason = agent.invoke_tool(sess_alice, tool)
    print(f'  [{"ALLOW" if ok else "DENY "}] {tool}')

print('Bob session:')
for tool in ['read_crm','search_web','read_calendar']:
    ok, reason = agent.invoke_tool(sess_bob, tool)
    print(f'  [{"ALLOW" if ok else "DENY "}] {tool}')

print()
print(f'Alice summary: {agent.session_summary(sess_alice)}')
print(f'Bob summary:   {agent.session_summary(sess_bob)}')
print('Key: sessions are fully isolated — alice cannot call crm tools, bob cannot call calendar')
